In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from kaggle_secrets import UserSecretsClient

# Достаем ваш токен из секретов Каггла
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Аналитик во мне подсказывает: проверьте точное название модели на HF. 
# Если Qwen3 еще "в пути", попробуйте Qwen/Qwen2.5-7B-Instruct
model_name = "Qwen/Qwen2.5-7B-Instruct" 

print("Загружаем токенайзер... Держим кулачки.")
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)

print("Тянем модель. Это может занять время, заварите чаю.")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto", # Пусть сам решит: float16 или bfloat16
    device_map="auto",
    token=hf_token
)

prompt = "Напиши короткий стих о том, почему код не запускается с первого раза."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

print("Генерируем ответ...")
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("-" * 30)
print(response)

Загружаем токенайзер... Держим кулачки.


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Генерируем ответ...
------------------------------
system
You are a helpful assistant.
user
Напиши короткий стих о том, почему код не запускается с первого раза.
assistant
Почему код не запускается с первого раза?
Он просит помощи у разума,
Чтобы найти ошибку и исправить,
Иначе бы не радовал глаз.
